In [1]:
from pyspark.sql.functions import col

# --- PART 1: Create Dimension Tables (The Context) ---
print("🏆 Building Gold Dimensions...")

# 1. DIM_CUSTOMERS
print("Fixing Dim Customers Uniqueness and Column Names...")
df_cust = spark.read.table("silver_customers")
df_dim_cust = df_cust.select(
    col("customer_unique_id").alias("CustomerKey"),
    col("city").alias("City"),       # FIXED: Changed from customer_city
    col("state").alias("State"),     # FIXED: Changed from customer_state
    col("zip_code").alias("ZipCode") # FIXED: Changed from customer_zip_code_prefix
).dropDuplicates(["CustomerKey"]) 

df_dim_cust.write.format("delta").mode("overwrite").saveAsTable("dim_customers")


# 2. DIM_PRODUCTS
df_prod = spark.read.table("silver_products")
df_dim_prod = df_prod.select(
    col("prod_id").alias("ProductKey"), # Note: If this fails next, check if it should be 'product_id'
    col("product_category_name").alias("Category"),
    col("product_weight_g").alias("Weight_g"),
    col("product_length_cm").alias("Length_cm")
)
df_dim_prod.write.format("delta").mode("overwrite").saveAsTable("dim_products")

# 3. DIM_SELLERS
df_sel = spark.read.table("silver_sellers")
df_dim_sel = df_sel.select(
    col("seller_id").alias("SellerKey"),
    col("seller_city").alias("SellerCity"),
    col("seller_state").alias("SellerState")
)
df_dim_sel.write.format("delta").mode("overwrite").saveAsTable("dim_sellers")

print("✅ Dimensions Created: Customers, Products, Sellers")

# --- PART 2: Create Fact Table (The Transactions) ---
print("Building Fact Sales with strict Silver compliance...")

# Load necessary Silver tables
df_orders = spark.read.table("silver_orders")
df_items = spark.read.table("silver_order_items")
df_bridge = spark.read.table("silver_customers") 

# JOIN LOGIC: Orders + Items + Bridge
df_fact = df_orders.join(df_items, "order_id", "inner") \
    .join(df_bridge, "customer_id", "inner") \
    .select(
        col("order_id").alias("OrderKey"),
        col("customer_unique_id").alias("CustomerKey"),
        col("product_id").alias("ProductKey"),          
        col("seller_id").alias("SellerKey"),            
        col("order_purchase_timestamp").alias("OrderDate"),
        col("price").alias("SalesAmount"),
        col("freight_value").alias("Freight"),
        col("order_status").alias("OrderStatus")
    )

df_fact.write.format("delta").mode("overwrite").saveAsTable("fact_sales")

print(f"✅ Fact Table Created: {df_fact.count()} Rows")

StatementMeta(, 73fd453e-6f2e-40e4-b62b-48ce58ab1c28, 3, Finished, Available, Finished, False)

🏆 Building Gold Dimensions...
Fixing Dim Customers Uniqueness and Column Names...
✅ Dimensions Created: Customers, Products, Sellers
Building Fact Sales with strict Silver compliance...
✅ Fact Table Created: 112650 Rows


In [2]:
from pyspark.sql.functions import col

print("🩹 Patching Silver Customers with Schema Overwrite...")

# 1. Read from the raw Bronze layer
df_bronze_cust = spark.read.table("bronze_customers")

# 2. Select the columns (matching your required schema)
df_silver_cust_fixed = df_bronze_cust.select(
    col("customer_id"),                  
    col("customer_unique_id"),           
    col("customer_city").alias("city"),  
    col("customer_state").alias("state"),
    col("customer_zip_code_prefix").alias("zip_code")
)

# 3. Overwrite the table AND the schema
df_silver_cust_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers") # <-- This will now succeed!

print("✅ Silver Customers Patched and Schema Updated!")

StatementMeta(, 73fd453e-6f2e-40e4-b62b-48ce58ab1c28, 4, Finished, Available, Finished, False)

🩹 Patching Silver Customers with Schema Overwrite...
✅ Silver Customers Patched and Schema Updated!


In [3]:
# Show me all the tables registered in this Lakehouse
display(spark.sql("SHOW TABLES"))

StatementMeta(, 73fd453e-6f2e-40e4-b62b-48ce58ab1c28, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d704676e-b24a-4c19-881f-55150c32e70a)